# D — Huy | Isolation Forest & LOF
**Trạng thái:** thí nghiệm trên Validation, **chưa phải kết quả cuối cùng**. Notebook chạy từ trên xuống dưới; `Class` chỉ dùng để chia phân tầng và chấm điểm, không dùng làm feature của hai mô hình không giám sát. Dataset: `data/raw/creditcard.csv` tại máy cá nhân. Không commit dữ liệu gốc.

## 1. Cài đặt, dữ liệu và kiểm tra chất lượng
Tìm thư mục gốc tự động; không phụ thuộc notebook được mở từ `Data_Mining/` hay `notebooks/`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score

RANDOM_STATE = 42
TOP_K = 100
FEATURES = [f"V{i}" for i in range(1, 29)]
root = Path.cwd().resolve()
while not (root / "requirements.txt").is_file():
    if root == root.parent:
        raise FileNotFoundError("Không tìm thấy thư mục gốc có requirements.txt")
    root = root.parent
csv_path = root / "data" / "raw" / "creditcard.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"Cần đặt creditcard.csv tại: {csv_path}")
df = pd.read_csv(csv_path)
required = set(FEATURES + ["Time", "Amount", "Class"])
if not required.issubset(df.columns):
    raise ValueError(f"Thiếu các cột: {sorted(required - set(df.columns))}")
y = df["Class"].astype(int)
print("Dataset:", df.shape, "| Missing:", int(df.isna().sum().sum()), "| Duplicates:", int(df.duplicated().sum()))
print("Phân bố nhãn:\n", y.value_counts().sort_index())
print("Root:", root)

Dataset: (284807, 31) | Missing: 0 | Duplicates: 1081
Phân bố nhãn:
 Class
0    284315
1       492
Name: count, dtype: int64
Root: C:\Users\Admin\Downloads\Data_Mining


## 2. Chia dữ liệu (bản thử nghiệm)
Chia phân tầng 70/15/15. **Cảnh báo:** dòng trùng có thể xuất hiện qua các tập; cần Cường và Mong phê duyệt phương pháp chia hợp lệ trước khi đánh giá Test. Tập Test được giữ nguyên, **không chấm điểm trong notebook bản nháp này**.

In [2]:
indices = np.arange(len(df))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=y.iloc[idx_temp], random_state=RANDOM_STATE)
for name, idx in [("Train", idx_train), ("Validation", idx_val), ("Test (chưa dùng)", idx_test)]:
    print(f"{name}: {len(idx):,} | fraud: {int(y.iloc[idx].sum())} | rate: {y.iloc[idx].mean():.4%}")

def cross_split_overlap(a, b):
    left = df.iloc[a].drop_duplicates()
    right = df.iloc[b]
    return len(right.merge(left, on=list(df.columns), how="inner"))
print("Trùng Train–Validation:", cross_split_overlap(idx_train, idx_val))
print("Trùng Train–Test:", cross_split_overlap(idx_train, idx_test))
print("Trùng Validation–Test:", cross_split_overlap(idx_val, idx_test))

X_train = df.iloc[idx_train][FEATURES]
X_val = df.iloc[idx_val][FEATURES]
y_val = y.iloc[idx_val].to_numpy()
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_val.to_numpy()).all()

Train: 199,364 | fraud: 344 | rate: 0.1725%
Validation: 42,721 | fraud: 74 | rate: 0.1732%
Test (chưa dùng): 42,722 | fraud: 74 | rate: 0.1732%
Trùng Train–Validation: 207
Trùng Train–Test: 249
Trùng Validation–Test: 61


## 3. Hàm chấm điểm chung
Điểm bất thường được đổi dấu để **cao hơn = bất thường hơn**. Precision@100 ở đây tính trên **toàn bộ Validation**, chưa phải ngân sách 100 giao dịch/ngày. AP là `average_precision_score`, không phải diện tích PR nội suy hình thang.

In [3]:
def build_results(indices, labels, scores):
    scores = np.asarray(scores)
    if len(indices) != len(labels) or len(labels) != len(scores):
        raise ValueError("Kích thước indices, labels và scores không khớp")
    out = pd.DataFrame({"original_index": indices, "anomaly_score": scores, "is_fraud": labels})
    out = out.sort_values(["anomaly_score", "original_index"], ascending=[False, True]).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out

def evaluate(results, name, k=TOP_K):
    k = min(k, len(results))
    tp = int(results.head(k)["is_fraud"].sum())
    fraud = int(results["is_fraud"].sum())
    return {"Model": name, "TP": tp, "FP": k - tp, "FN": fraud - tp,
            "Precision@100": tp / k, "Recall@100": tp / fraud if fraud else float("nan"),
            "AP": average_precision_score(results["is_fraud"], results["anomaly_score"])}

## 4. Isolation Forest — không giám sát
Huấn luyện với tất cả mẫu Train; scaler chỉ `fit` trên Train. Dùng V1–V28 để giữ lại cấu hình thí nghiệm gốc của Huy.

In [4]:
iso_demo = Pipeline([
    ("scaler", StandardScaler()),
    ("model", IsolationForest(n_estimators=100, contamination="auto", random_state=RANDOM_STATE, n_jobs=-1))
])
iso_demo.fit(X_train)
iso_scores = -iso_demo.decision_function(X_val)
val_results = build_results(idx_val, y_val, iso_scores)
print("=== ISOLATION FOREST ===")
display(pd.DataFrame([evaluate(val_results, "Isolation Forest")]))

=== ISOLATION FOREST ===


,Model,TP,FP,FN,Precision@100,Recall@100,AP
0,Isolation Forest,24,76,50,0.24,0.324324,0.183223


## 5. LOF — không giám sát, chọn mẫu Train không dùng `Class`
LOF `novelty=True` để chấm điểm các giao dịch mới (Validation). Giữ cố định mẫu Train 20.000 giao dịch khi thử `n_neighbors` để so sánh tham số công bằng. Không chọn mẫu bình thường bằng nhãn cho mô hình chính.

In [5]:
rng = np.random.default_rng(RANDOM_STATE)
sample_idx_unsupervised = rng.choice(idx_train, size=min(20000, len(idx_train)), replace=False)
X_train_lof = df.iloc[sample_idx_unsupervised][FEATURES]
lof_scaler = StandardScaler()
X_train_lof_scaled = lof_scaler.fit_transform(X_train_lof)
X_val_lof_scaled = lof_scaler.transform(X_val)

lof_tuning_rows = []
lof_score_by_n = {}
for neighbors in [10, 20, 50, 100]:
    model = LocalOutlierFactor(n_neighbors=neighbors, novelty=True, n_jobs=-1)
    model.fit(X_train_lof_scaled)
    scores = -model.decision_function(X_val_lof_scaled)
    lof_score_by_n[neighbors] = scores
    result = build_results(idx_val, y_val, scores)
    metrics = evaluate(result, f"LOF n={neighbors}")
    lof_tuning_rows.append({"n_neighbors": neighbors, "Fraud Top 100": metrics["TP"],
                            "Precision@100": metrics["Precision@100"],
                            "Recall@100": metrics["Recall@100"], "AP": metrics["AP"]})
tuning_df = pd.DataFrame(lof_tuning_rows)
display(tuning_df)

best_n = int(tuning_df.sort_values(["Precision@100", "AP"], ascending=False).iloc[0]["n_neighbors"])
lof_results_best = build_results(idx_val, y_val, lof_score_by_n[best_n])
print("Best n_neighbors trong 4 giá trị đã thử:", best_n)

,n_neighbors,Fraud Top 100,Precision@100,Recall@100,AP
0,10,3,0.03,0.040541,0.005698
1,20,3,0.03,0.040541,0.013411
2,50,31,0.31,0.418919,0.191042
3,100,39,0.39,0.527027,0.326613


Best n_neighbors trong 4 giá trị đã thử: 100


## 6. Chẩn đoán phụ: LOF dùng nhãn Train để chọn mẫu bình thường
Phần này tái lập thí nghiệm khám phá ban đầu, **không được xem là LOF hoàn toàn không giám sát**. Không dùng kết quả này làm mô hình chính hoặc thay thế kết quả Test.

In [6]:
normal_idx = idx_train[y.iloc[idx_train].to_numpy() == 0]
normal_sample_idx = np.random.default_rng(RANDOM_STATE).choice(normal_idx, size=min(20000, len(normal_idx)), replace=False)
lof_normal_only = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=-1))
])
lof_normal_only.fit(df.iloc[normal_sample_idx][FEATURES])
label_filtered_results = build_results(idx_val, y_val, -lof_normal_only.decision_function(X_val))
display(pd.DataFrame([evaluate(label_filtered_results, "LOF lọc nhãn train (tham khảo)")]))

,Model,TP,FP,FN,Precision@100,Recall@100,AP
0,LOF lọc nhãn train (tham khảo),49,51,25,0.49,0.662162,0.421065


## 7. Random Baseline — 100 lần
Dùng 100 seed để ghi lại dao động của baseline ngẫu nhiên; không dựa vào một lần bốc mẫu duy nhất.

In [7]:
baseline_rows = []
for seed in range(100):
    random_scores = np.random.default_rng(seed).random(len(y_val))
    baseline_result = build_results(idx_val, y_val, random_scores)
    metrics = evaluate(baseline_result, "Random baseline")
    baseline_rows.append({"seed": seed, "fraud_top100": metrics["TP"],
                          "precision_at_100": metrics["Precision@100"],
                          "recall_at_100": metrics["Recall@100"], "ap": metrics["AP"]})
baseline_100 = pd.DataFrame(baseline_rows)
baseline_summary = baseline_100[["fraud_top100", "precision_at_100", "recall_at_100", "ap"]].agg(["mean", "std"])
display(baseline_summary)
print("Phân bố fraud_top100:\n", baseline_100["fraud_top100"].value_counts().sort_index())

,fraud_top100,precision_at_100,recall_at_100,ap
mean,0.15000,0.001500,0.002027,0.001959
std,0.35887,0.003589,0.004850,0.000518


Phân bố fraud_top100:
 fraud_top100
0    85
1    15
Name: count, dtype: int64


## 8. Error Analysis và bảng so sánh
TP/FP/FN được định nghĩa **theo ngưỡng Top 100**. Không nhầm với nhãn phân lớp của model. Bảng chính chỉ so sánh hai mô hình không giám sát.

In [8]:
comparison_errors = pd.DataFrame([
    evaluate(val_results, "Isolation Forest"),
    evaluate(lof_results_best, f"LOF (n={best_n})")
])
display(comparison_errors)
for name, result in [("Isolation Forest", val_results), (f"LOF (n={best_n})", lof_results_best)]:
    print(f"{name}: TP={int(result.head(TOP_K)['is_fraud'].sum())}, "
          f"FP={TOP_K - int(result.head(TOP_K)['is_fraud'].sum())}, "
          f"FN={int(result['is_fraud'].sum()) - int(result.head(TOP_K)['is_fraud'].sum())}")

,Model,TP,FP,FN,Precision@100,Recall@100,AP
0,Isolation Forest,24,76,50,0.24,0.324324,0.183223
1,LOF (n=100),39,61,35,0.39,0.527027,0.326613


Isolation Forest: TP=24, FP=76, FN=50
LOF (n=100): TP=39, FP=61, FN=35


## 9. Xuất các file nháp
Xuất CSV trong `outputs/`. Không xuất nhãn Test hoặc giả lập score Test. Những file dưới đây vẫn có hậu tố `_draft` để tránh bị hiểu là kết quả chính thức.

In [9]:
output_dir = root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
tuning_df.to_csv(output_dir / "lof_tuning_validation_draft.csv", index=False)
baseline_100.to_csv(output_dir / "random_baseline_100_runs_draft.csv", index=False)
baseline_summary.to_csv(output_dir / "random_baseline_summary_draft.csv")
comparison_errors.to_csv(output_dir / "model_error_analysis_draft.csv", index=False)
print("Đã lưu 4 file đánh giá nháp vào:", output_dir)

Đã lưu 4 file đánh giá nháp vào: C:\Users\Admin\Downloads\Data_Mining\outputs


## 10. Kết luận và điều kiện hoàn thành
- Đây là thí nghiệm sơ bộ, dữ liệu Validation có dòng trùng với Train. **Không kết luận kết quả cuối cùng từ các con số này.**
- LOF sử dụng 20.000 mẫu train, Isolation Forest sử dụng toàn bộ train nên so sánh chưa hoàn toàn đồng nhất.
- `V1–V28` mới là đặc trưng thử nghiệm; chưa tích hợp `Time` và `Amount` đã xử lý bởi Cường.
- Top 100 trên Validation chưa tương ứng câu hỏi nghiệp vụ 100 giao dịch **mỗi ngày**.
- **Việc cần làm sau khi Cường/Mong thống nhất pipeline:** tách train/validation/test không bị leakage, huấn luyện lại, chọn tham số trên Validation, dùng Test đúng một lần để báo cáo, xuất `anomaly_scores_test.csv`, cập nhật decision log.

**Thử tái lập:** trong VS Code chọn **Restart Kernel and Run All Cells**. File này được tổ chức để các ô có thể chạy theo thứ tự, nhưng cần có `creditcard.csv` trên máy; chưa được xác minh bằng lần chạy end-to-end trong môi trường tạo file.

In [10]:
import pandas as pd
from pathlib import Path

root = Path.cwd()
if not (root / "data").exists():
    root = root.parent

data_path = root / "data/processed/transactions_processed.csv"

processed_df = pd.read_csv(data_path)

print("Kích thước:", processed_df.shape)
print("\nDanh sách cột:")
print(processed_df.columns.tolist())
print("\n5 dòng đầu:")
display(processed_df.head())

Kích thước: (284807, 47)

Danh sách cột:
['txn_id', 'time_sec', 'rel_day', 'rel_hour', 'hour_of_cycle', 'hour_cycle_sin', 'hour_cycle_cos', 'amount', 'amount_log1p', 'amount_band_id', 'amount_band', 'is_zero_amount', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'is_fraud', 'is_duplicate', 'dup_of_txn_id', 'in_model_set', 'split_temporal', 'split_random', 'cv_fold']

5 dòng đầu:


,txn_id,time_sec,rel_day,rel_hour,hour_of_cycle,hour_cycle_sin,hour_cycle_cos,amount,amount_log1p,amount_band_id,...,V26,V27,V28,is_fraud,is_duplicate,dup_of_txn_id,in_model_set,split_temporal,split_random,cv_fold
0,1,0,0,0,0,0.000000,1.0,149.62,5.014760,4,...,-0.189115,0.133558,-0.021053,0,0,1,1,train,val,1
1,2,0,0,0,0,0.000000,1.0,2.69,1.305626,1,...,0.125895,-0.008983,0.014724,0,0,2,1,train,train,3
2,3,1,0,0,0,0.000073,1.0,378.66,5.939276,4,...,-0.139097,-0.055353,-0.059752,0,0,3,1,train,train,4
3,4,1,0,0,0,0.000073,1.0,123.50,4.824306,4,...,-0.221929,0.062723,0.061458,0,0,4,1,train,val,0
4,5,2,0,0,0,0.000145,1.0,69.99,4.262539,3,...,0.502292,0.219422,0.215153,0,0,5,1,train,train,4


In [11]:

# Kiểm tra schema dữ liệu của Cường

print("=== TẤT CẢ CÁC CỘT ===")
for col in processed_df.columns:
    print(col)

print("\n=== CÁC CỘT LIÊN QUAN ĐẾN SPLIT / LABEL ===")
for col in processed_df.columns:
    if any(x in col.lower() for x in
           ["split", "train", "test", "label",
            "fraud", "duplicate", "feature"]):
        print(col)

print("\n=== SỐ GIÁ TRỊ THIẾU ===")
print(processed_df.isna().sum().sum())


=== TẤT CẢ CÁC CỘT ===
txn_id
time_sec
rel_day
rel_hour
hour_of_cycle
hour_cycle_sin
hour_cycle_cos
amount
amount_log1p
amount_band_id
amount_band
is_zero_amount
V1
V2
V3
V4
V5
V6
V7
V8
V9
V10
V11
V12
V13
V14
V15
V16
V17
V18
V19
V20
V21
V22
V23
V24
V25
V26
V27
V28
is_fraud
is_duplicate
dup_of_txn_id
in_model_set
split_temporal
split_random
cv_fold

=== CÁC CỘT LIÊN QUAN ĐẾN SPLIT / LABEL ===
is_fraud
is_duplicate
split_temporal
split_random

=== SỐ GIÁ TRỊ THIẾU ===
0


In [12]:
print("=== DANH SÁCH 47 CỘT ===")
print("\n".join(processed_df.columns.tolist()))

print("\n=== PHÂN BỐ SPLIT ===")
print(processed_df["split_random"].value_counts())

print("\n=== CÁC CỘT LIÊN QUAN ĐẾN MÔ HÌNH ===")
print([
    c for c in processed_df.columns
    if any(x in c.lower() for x in
           ["class", "fraud", "duplicate", "split", "scaled"])
])

=== DANH SÁCH 47 CỘT ===
txn_id
time_sec
rel_day
rel_hour
hour_of_cycle
hour_cycle_sin
hour_cycle_cos
amount
amount_log1p
amount_band_id
amount_band
is_zero_amount
V1
V2
V3
V4
V5
V6
V7
V8
V9
V10
V11
V12
V13
V14
V15
V16
V17
V18
V19
V20
V21
V22
V23
V24
V25
V26
V27
V28
is_fraud
is_duplicate
dup_of_txn_id
in_model_set
split_temporal
split_random
cv_fold

=== PHÂN BỐ SPLIT ===
split_random
train                 170235
test                   56746
val                    56745
excluded_duplicate      1081
Name: count, dtype: int64

=== CÁC CỘT LIÊN QUAN ĐẾN MÔ HÌNH ===
['is_fraud', 'is_duplicate', 'split_temporal', 'split_random']


In [13]:
print("=== RANDOM SPLIT ===")
print(processed_df["split_random"].value_counts().to_string())

print("\n=== TEMPORAL SPLIT ===")
print(processed_df["split_temporal"].value_counts().to_string())

print("\n=== DUPLICATES THEO RANDOM SPLIT ===")
print(pd.crosstab(
    processed_df["split_random"],
    processed_df["is_duplicate"]
).to_string())

=== RANDOM SPLIT ===
split_random
train                 170235
test                   56746
val                    56745
excluded_duplicate      1081

=== TEMPORAL SPLIT ===
split_temporal
test                  139490
train                 115388
val                    28848
excluded_duplicate      1081

=== DUPLICATES THEO RANDOM SPLIT ===
is_duplicate             0     1
split_random                    
excluded_duplicate       0  1081
test                 56746     0
train               170235     0
val                  56745     0


In [14]:

# Sử dụng cách chia dữ liệu của Cường
# Không tự chia lại bằng train_test_split

train_df = processed_df[
    processed_df["split_random"] == "train"
].copy()

val_df = processed_df[
    processed_df["split_random"] == "val"
].copy()

test_df = processed_df[
    processed_df["split_random"] == "test"
].copy()

# Chỉ dùng V1-V28 để tạo baseline tương đương thử nghiệm cũ
feature_cols = [f"V{i}" for i in range(1, 29)]

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

# Nhãn chỉ dùng khi đánh giá
y_val = val_df["is_fraud"]

print("=== MODEL DATA READY ===")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", len(test_df))
print("Fraud trong Validation:", int(y_val.sum()))
print("Có giá trị thiếu:", X_train.isna().sum().sum())


=== MODEL DATA READY ===
Train: (170235, 28)
Validation: (56745, 28)
Test: 56746
Fraud trong Validation: 94
Có giá trị thiếu: 0


In [15]:

# Isolation Forest - Dữ liệu tiền xử lý của Cường
# Đánh giá trên Validation, chưa sử dụng Test

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
import numpy as np

iso_final = Pipeline([
    ("scaler", StandardScaler()),
    ("model", IsolationForest(
        n_estimators=100,
        contamination="auto",
        random_state=42,
        n_jobs=-1
    ))
])

# Huấn luyện không sử dụng nhãn gian lận
iso_final.fit(X_train)

# Tính anomaly score trên Validation
iso_scores = -iso_final.decision_function(X_val)

# Chọn 100 giao dịch có điểm bất thường cao nhất
top100_idx = np.argsort(-iso_scores)[:100]
y_val_array = y_val.to_numpy()

fraud_detected = int(y_val_array[top100_idx].sum())

precision = fraud_detected / 100
recall = fraud_detected / int(y_val_array.sum())
ap = average_precision_score(y_val_array, iso_scores)

print("=== ISOLATION FOREST - PROCESSED DATA ===")
print("Fraud Top 100:", fraud_detected)
print(f"Precision@100: {precision:.2%}")
print(f"Recall@100: {recall:.2%}")
print(f"Average Precision: {ap:.4f}")


=== ISOLATION FOREST - PROCESSED DATA ===
Fraud Top 100: 29
Precision@100: 29.00%
Recall@100: 30.85%
Average Precision: 0.2674


In [16]:

# LOF - Dữ liệu tiền xử lý của Cường
# Không dùng nhãn is_fraud để chọn mẫu train

from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
import numpy as np

rng = np.random.default_rng(42)

sample_idx = rng.choice(
    len(X_train),
    size=min(20000, len(X_train)),
    replace=False
)

X_train_lof = X_train.iloc[sample_idx]

scaler_lof = StandardScaler()
X_train_scaled = scaler_lof.fit_transform(X_train_lof)
X_val_scaled = scaler_lof.transform(X_val)

lof_model = LocalOutlierFactor(
    n_neighbors=100,
    novelty=True,
    n_jobs=-1
)

lof_model.fit(X_train_scaled)

lof_scores = -lof_model.decision_function(X_val_scaled)

y_val_array = y_val.to_numpy()
top100_idx = np.argsort(-lof_scores)[:100]

fraud_detected = int(y_val_array[top100_idx].sum())
total_fraud = int(y_val_array.sum())

print("=== LOF - PROCESSED DATA ===")
print("Fraud Top 100:", fraud_detected)
print(f"Precision@100: {fraud_detected / 100:.2%}")
print(f"Recall@100: {fraud_detected / total_fraud:.2%}")
print(
    "Average Precision:",
    round(average_precision_score(y_val_array, lof_scores), 4)
)


=== LOF - PROCESSED DATA ===
Fraud Top 100: 49
Precision@100: 49.00%
Recall@100: 52.13%
Average Precision: 0.4708


In [17]:

import pandas as pd
from pathlib import Path

# Tổng hợp kết quả Validation
comparison = pd.DataFrame([
    {
        "Model": "Isolation Forest",
        "Fraud_Top100": 29,
        "Precision@100": 0.29,
        "Recall@100": 29 / 94,
        "AP": 0.2674
    },
    {
        "Model": "LOF (n=100)",
        "Fraud_Top100": 49,
        "Precision@100": 0.49,
        "Recall@100": 49 / 94,
        "AP": 0.4708
    }
])

output_dir = root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    output_dir / "model_comparison_processed_validation.csv",
    index=False
)

display(comparison)
print("Đã lưu bảng so sánh Validation!")


,Model,Fraud_Top100,Precision@100,Recall@100,AP
0,Isolation Forest,29,0.29,0.308511,0.2674
1,LOF (n=100),49,0.49,0.521277,0.4708


Đã lưu bảng so sánh Validation!


In [18]:
results = []

for n in [20, 50, 100, 150, 200]:
    model = LocalOutlierFactor(
        n_neighbors=n,
        novelty=True,
        n_jobs=-1
    )

    model.fit(X_train_scaled)
    scores = -model.decision_function(X_val_scaled)

    top100 = np.argsort(-scores)[:100]
    fraud = int(y_val_array[top100].sum())

    results.append({
        "n_neighbors": n,
        "Fraud_Top100": fraud,
        "Precision@100": fraud / 100,
        "Recall@100": fraud / int(y_val_array.sum()),
        "AP": average_precision_score(y_val_array, scores)
    })

lof_tuning = pd.DataFrame(results)
display(lof_tuning)

,n_neighbors,Fraud_Top100,Precision@100,Recall@100,AP
0,20,2,0.02,0.021277,0.018422
1,50,45,0.45,0.478723,0.385786
2,100,49,0.49,0.521277,0.470816
3,150,46,0.46,0.489362,0.443241
4,200,43,0.43,0.457447,0.417016


In [19]:
lof_tuning.to_csv(
    output_dir / "lof_tuning_processed_validation.csv",
    index=False
)

print("Đã lưu kết quả tuning LOF!")

Đã lưu kết quả tuning LOF!


In [20]:
# Chuẩn bị Test theo cách chia dữ liệu của Cường
X_test = test_df[feature_cols]
y_test = test_df["is_fraud"].to_numpy()

# Tính điểm bất thường bằng các mô hình đã huấn luyện
iso_test_scores = -iso_final.decision_function(X_test)
lof_test_scores = -lof_model.decision_function(
    scaler_lof.transform(X_test)
)

def evaluate_test(scores, labels):
    top100 = np.argsort(-scores)[:100]
    fraud = int(labels[top100].sum())

    return {
        "Fraud_Top100": fraud,
        "Precision@100": fraud / 100,
        "Recall@100": fraud / int(labels.sum()),
        "AP": average_precision_score(labels, scores)
    }

test_results = pd.DataFrame({
    "Isolation Forest": evaluate_test(iso_test_scores, y_test),
    "LOF (n=100)": evaluate_test(lof_test_scores, y_test)
}).T

print("=== TEST RESULTS ===")
print("Số giao dịch gian lận trong Test:", int(y_test.sum()))
display(test_results)

=== TEST RESULTS ===
Số giao dịch gian lận trong Test: 95


,Fraud_Top100,Precision@100,Recall@100,AP
Isolation Forest,27.0,0.27,0.284211,0.160945
LOF (n=100),38.0,0.38,0.400000,0.312462


In [21]:
# Xuất điểm bất thường của hai mô hình trên Test

scores_test = pd.DataFrame({
    "txn_id": test_df["txn_id"].to_numpy(),
    "is_fraud": y_test,
    "split": "test",
    "isolation_forest_score": iso_test_scores,
    "lof_score": lof_test_scores
})

# Xếp hạng: điểm bất thường càng cao, hạng càng gần 1
scores_test["isolation_forest_rank"] = (
    scores_test["isolation_forest_score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

scores_test["lof_rank"] = (
    scores_test["lof_score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

scores_test.to_csv(
    output_dir / "anomaly_scores_test.csv",
    index=False
)

print("Đã xuất:", output_dir / "anomaly_scores_test.csv")
print("Số dòng:", len(scores_test))
display(scores_test.head())

Đã xuất: c:\Users\Admin\Downloads\Data_Mining\outputs\anomaly_scores_test.csv
Số dòng: 56746


,txn_id,is_fraud,split,isolation_forest_score,lof_score,isolation_forest_rank,lof_rank
0,6,0,test,-0.147313,-0.521742,56578,55491
1,10,0,test,-0.139221,-0.454150,53898,38416
2,11,0,test,-0.109527,-0.511167,30778,52357
3,14,0,test,-0.129507,-0.458316,46694,39236
4,16,0,test,-0.103510,-0.419997,26755,32634


In [22]:
# Phân tích Top 100 trên Test
error_results = []

for name, scores in [
    ("Isolation Forest", iso_test_scores),
    ("LOF", lof_test_scores)
]:
    top100 = np.argsort(-scores)[:100]
    tp = int(y_test[top100].sum())
    fp = 100 - tp
    fn = int(y_test.sum()) - tp

    error_results.append({
        "Model": name,
        "TP": tp,
        "FP": fp,
        "FN": fn
    })

error_analysis = pd.DataFrame(error_results)

display(error_analysis)

error_analysis.to_csv(
    output_dir / "model_error_analysis_processed_test.csv",
    index=False
)

,Model,TP,FP,FN
0,Isolation Forest,27,73,68
1,LOF,38,62,57
